# Scene segmentation and reconstruction with GCPR16

[中文说明](#中文说明) · [Backend guide](../asset/gcpr16_backend.md)

Follow the original scene notebook: load the point cloud, inspect boxes, load pretrained matrices, segment vehicles, alternate pose/shape fitting, and display meshes with rooftop vertices. Select the **Python (shapepriors-gcpr16)** kernel and run cells in order. Open3D windows block the current cell until closed.

## 中文说明

流程与原场景 Notebook 对应：加载点云、查看框、直接加载预训练矩阵、框内分割、交替拟合位姿与形状，最后显示网格和车顶。选择 **Python (shapepriors-gcpr16)** 内核后顺序运行。Open3D 窗口关闭后，当前单元才继续。

Use Conda environment `shapepriors-gcpr16`. Install notebook dependencies with `conda run -n shapepriors-gcpr16 python -m pip install -r requirements-gcpr16-notebook.txt` from the repository root. Fitting runs on CPU float64 even if output tensors use CUDA. Set `VISUALIZE=False` for unattended execution.

使用 Conda 环境 `shapepriors-gcpr16`，在仓库根目录执行上方命令安装 Notebook 依赖。即使输出张量使用 CUDA，拟合也在 CPU 双精度上执行。无人值守运行时将 `VISUALIZE=False`。

In [ ]:
from pathlib import Path
from collections import Counter
from time import perf_counter
import sys

import numpy as np
import torch

# Resolve paths from either the repository root or tools/.
# 从仓库根目录或 tools/ 启动均可定位项目。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'vehicle_reconstruction_gcpr16').is_dir()
             and (p / 'tools').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start this notebook from the repository root or tools/. / 请从仓库根目录或 tools/ 启动。')
for directory in (ROOT, ROOT / 'tools'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from vehicle_reconstruction_gcpr16 import vehicle_reconstructor, vehicle_object
from vehicle_reconstruction_gcpr16.scene_segmentation import point_cloud_scene
from vehicle_reconstruction_gcpr16.structures import as_numpy

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float64
VISUALIZE = globals().get('VISUALIZE', True)

POINTS_PATH = ROOT / 'data' / '00043_pts.npy'
BOXES_PATH = ROOT / 'data' / '00043_gt.npy'
MATRICES_DIR = ROOT / 'data' / 'matrices'

# Original eligibility settings; for sparse LiDAR try MIN_POINTS=11, MAX_DISTANCE=None.
# 原版筛选参数；稀疏 LiDAR 可尝试 MIN_POINTS=11、MAX_DISTANCE=None。
MIN_POINTS = 5
MAX_DISTANCE = 30.0
OUTER_ITERATIONS = 10
SOLVER_ITERATIONS = 100
SMOOTHING = False

print(f'Project / 项目: {ROOT}')
print(f'Tensor device / 张量设备: {DEVICE}; solver / 求解器: CPU float64')

## Loading point cloud / 加载点云

Points are `[N,4]` (xyz, intensity). Boxes are `[B,7]`: center xyz, length, width, height, yaw. Coordinates are meters in LiDAR axes (+x forward, +y left, +z up).

点云格式为 xyz 加强度；框为几何中心 xyz、长、宽、高、朝向。单位为米，采用 LiDAR 坐标（前、左、上）。

In [ ]:
pts = np.load(POINTS_PATH, allow_pickle=False).reshape(-1, 4).astype(np.float64)
gt_box = np.load(BOXES_PATH, allow_pickle=False).reshape(-1, 7).astype(np.float64)
if not np.isfinite(pts).all() or not np.isfinite(gt_box).all():
    raise ValueError('Point cloud and boxes must be finite. / 点云和框必须为有限值。')
print('Points / 点云:', pts.shape)
print('Boxes / 框:', gt_box.shape)

## Visualizing input / 查看输入

White points and blue initial boxes use the same drawing utility as the original notebook. Close the window to proceed. Set `VISUALIZE=False` in the first cell to skip windows.

使用原 Notebook 的绘图工具，白色点云、蓝色初始框。关闭窗口后继续；首个代码单元的 `VISUALIZE=False` 可跳过窗口。

In [ ]:
def show_scene_window(title, points, initial_boxes, optimized_boxes=None, geometry=None):
    import open3d as o3d
    from open3d_vis_utils import draw_scenes

    vis = o3d.visualization.Visualizer()
    try:
        if not vis.create_window(window_name=title, width=1280, height=800):
            raise RuntimeError('Open3D window creation failed. / Open3D 窗口创建失败。')
        draw_scenes(vis, points=points, gt_boxes=initial_boxes,
                    ref_boxes=optimized_boxes, draw_origin=True)
        if geometry is not None:
            vis.add_geometry(geometry)
        vis.run()
    finally:
        vis.destroy_window()

if VISUALIZE:
    show_scene_window('GCPR16 input / 输入', pts, gt_box)
else:
    print('Input window skipped / 已跳过输入窗口')

## Preparing the pretrained reconstructor / 加载预训练重建器

Read `S.bin`, `V.bin`, and `mean_shape.bin` directly. The fixed grid is 60×40×60, origin (-3,-3,-3), spacing 0.1 m. The first five PCA components define `shape = mean + V z`; no mesh preprocessing or PCA fitting is required.

直接读取三个原版矩阵，使用固定网格及前 5 个主成分，无需 ApolloScape、网格预处理或 PCA 训练。

In [ ]:
vehi_reconstructor = vehicle_reconstructor(
    matrices_dir=MATRICES_DIR, k=5, device=DEVICE, dtype=DTYPE,
)
print('Grid / 网格:', vehi_reconstructor.TSDF.resolution)
print('Origin / 原点:', as_numpy(vehi_reconstructor.TSDF.origin))
print('Unit / 间距:', as_numpy(vehi_reconstructor.unit))
print('S:', as_numpy(vehi_reconstructor._S))

## Segmenting scene vehicles / 框内车辆分割

Keep actual points inside each oriented box. The adapter converts LiDAR box centers to the original camera/bottom-center convention internally. `MIN_POINTS` is inclusive; `point_threshold=MIN_POINTS-1` preserves the API's strict `count > threshold` convention. The distance rule tests native forward translation, not Euclidean range.

保留有向三维框内的真实点，内部完成原版坐标转换。`MIN_POINTS` 表示最少点数，阈值传入时减 1，以适配严格大于约定。距离规则检查原版前向平移，并非欧氏距离；`MAX_DISTANCE=None` 可关闭。

In [ ]:
if MIN_POINTS < 1:
    raise ValueError('MIN_POINTS must be positive. / MIN_POINTS 必须为正整数。')
scene = point_cloud_scene(
    pts=torch.as_tensor(pts, device=DEVICE, dtype=DTYPE),
    gt_boxes=torch.as_tensor(gt_box, device=DEVICE, dtype=DTYPE),
    coordinate_frame='lidar',
    vehi_reconstructor=vehi_reconstructor,
)
vehicles: vehicle_object = scene.get_vehicles()
point_counts = as_numpy(vehicles.pts.num_points_per_cloud())
initial_translation = as_numpy(vehicles.translation).copy()
eligible = point_counts >= MIN_POINTS
if MAX_DISTANCE is not None:
    eligible &= vehicles.poses[:, 2] <= MAX_DISTANCE
print(f'Vehicles / 车辆: {len(vehicles)}')
print('Points per box / 每框点数:', point_counts)
print('Eligible box indices / 参与拟合的框索引:', np.flatnonzero(eligible))

## Alternating pose and shape fitting / 交替拟合位姿与形状

Each outer iteration solves pose (three translations and yaw), then original shape coefficients `z`. Original Huber/prior weights and boundary rules are retained. This backend uses CPU LM/IRLS instead of Ceres, with corrected analytic derivatives. Re-running this cell continues from the current estimate; re-run segmentation for a fresh start.

每轮先优化三个平移分量和朝向，再优化原版形状系数 `z`。保持原版损失及边界规则，采用修正解析梯度的 CPU LM/IRLS 求解。重复运行本单元会从当前估计继续；重新运行分割单元可重置。

Timing reports fitting wall time, cumulative solver time, mean time per attempted vehicle, and each vehicle's fit time. Fitting wall time excludes model loading, ROI extraction, mesh extraction and visualization; interface time also includes any lazy segmentation and input preparation.

计时显示拟合总耗时、求解器累计耗时、每辆参与求解车辆的平均耗时及逐车耗时。拟合总耗时不含权重加载、框内分割、网格提取和可视化；接口总耗时还包括可能发生的延迟分割与输入准备。

In [ ]:
started = perf_counter()
fit_results = scene.pose_estimate(
    vehi_reconstructor=vehi_reconstructor,
    iter=OUTER_ITERATIONS,
    point_threshold=MIN_POINTS - 1,
    max_solver_iterations=SOLVER_ITERATIONS,
    max_distance=MAX_DISTANCE,
)
fit_call_seconds = perf_counter() - started
fit_timing = scene.fit_timing
print(f'Fitting total / 拟合总耗时: {fit_timing["total_seconds"]:.6f} s')
print(f'Solver total / 求解器累计耗时: {fit_timing["solver_seconds"]:.6f} s')
print(f'Mean per vehicle / 平均每车: {fit_timing["mean_vehicle_seconds"] * 1000:.3f} ms '
      f'({fit_timing["vehicle_count"]} vehicles / 辆)')
print(f'pose_estimate call / 接口总耗时: {fit_call_seconds:.6f} s')
fitted_indices = np.array([i for i, result in enumerate(fit_results)
                           if result.get('success', False)], dtype=int)
print('Fitted box indices / 已拟合框索引:', fitted_indices)
print('Skipped / 跳过原因:', dict(Counter(result.get('skipped', 'solver_failure')
      for result in fit_results if not result.get('success', False))))
for index in fitted_indices:
    result = fit_results[index]
    print(f'box={index}, points={result["point_count"]}, '
          f'outer={len(result["history"])}, '
          f'fit_time={result["elapsed_seconds"] * 1000:.3f} ms, '
          f'shape_cost={result["final_costs"]["shape_cost"]:.6g}, '
          f'status={result["termination"]}')
    print('  xyz/yaw before / 优化前:', initial_translation[index])
    print('  xyz/yaw after  / 优化后:', as_numpy(vehicles.translation)[index])

training_point_clouds = vehicles.get_training_point_clouds()
fitted_training_points = as_numpy(training_point_clouds[fitted_indices].points_packed())
if len(fitted_training_points):
    print('Fitted grid-coordinate bounds / 已拟合点网格坐标范围:',
          fitted_training_points.min(axis=0), fitted_training_points.max(axis=0))

## Reconstructing meshes and rooftop vertices / 重建网格与车顶顶点

Extract the zero isosurface and place it using the optimized poses. Default output is unsmoothed. Rooftop vertices lie within 0.2 m of each vehicle's highest point. `reconstructed_indices` maps mesh order back to input box indices.

提取零等值面并按优化后位姿放回场景，默认不平滑。距每车最高点 0.2 米内的顶点标记为车顶。`reconstructed_indices` 将网格顺序映射到输入框索引。

In [ ]:
mesh, packed_rooftop_idx = vehicles.reconstruct_at_scene(
    vehi_reconstructor, threshold=MIN_POINTS - 1,
    smoothing=SMOOTHING, show_rooftop=True,
)
mesh_vertices = as_numpy(mesh.verts_packed())
mesh_faces = as_numpy(mesh.faces_packed()).astype(np.int32)
rooftop_mask = as_numpy(packed_rooftop_idx).astype(bool)
optimized_boxes = gt_box.copy()
optimized_boxes[:, [0, 1, 2, 6]] = as_numpy(vehicles.translation)
mesh_box_indices = vehicles.reconstructed_indices.copy()
vertex_colors = np.tile([0.5, 0.75, 0.5], (len(mesh_vertices), 1))
vertex_colors[rooftop_mask] = [0.6, 0.5, 0.5]
print('Mesh-to-box indices / 网格对应框索引:', mesh_box_indices)
print(f'Vertices / 顶点: {len(mesh_vertices)}; '
      f'triangles / 三角面: {len(mesh_faces)}; rooftop / 车顶顶点: {rooftop_mask.sum()}')

## Visualizing the reconstructed scene / 显示重建场景

Blue boxes are initial poses; green boxes show optimized poses for reconstructed vehicles. The original box dimensions are retained for comparison; the prior determines mesh dimensions. Meshes are green, rooftop vertices brown. If all vehicles are skipped, the input scene is still displayed.

蓝框为初始位姿，绿框为重建车辆优化后的位姿。对比框保留原尺寸，网格尺寸由先验决定。网格为绿色，车顶顶点为棕色；全部车辆跳过时仍可显示输入场景。

In [ ]:
def make_open3d_mesh(vertices, triangles, colors):
    import open3d as o3d
    geometry = o3d.geometry.TriangleMesh(
        o3d.utility.Vector3dVector(vertices),
        o3d.utility.Vector3iVector(triangles),
    )
    geometry.vertex_colors = o3d.utility.Vector3dVector(colors)
    geometry.compute_vertex_normals()
    return geometry

if VISUALIZE:
    mesh_o3d = make_open3d_mesh(mesh_vertices, mesh_faces, vertex_colors) if len(mesh_vertices) else None
    show_scene_window('GCPR16 reconstruction / 重建', pts, gt_box,
                      optimized_boxes=optimized_boxes[mesh_box_indices], geometry=mesh_o3d)
else:
    print('Reconstruction window skipped / 已跳过重建窗口')